# Week 3 Lab. Markov Chains and the Stationary Distribution
**Time Series Analysis & Random Processes** · Graduate School of Data Science, Chonnam National University · notebook v4 (2026-09-13, Week 2 connection)

---

### What this lab does

Last week asked when one long record can reveal a process. Today we answer that question with a finite-state Markov model.

Record Sunny as 1 and every other state as 0. The average of these numbers is exactly the sunny visit frequency.
Counting visits in this lab is last week's time average in a concrete form.

Stationarity gives stable quantities to study. The ACF describes linear dependence across time; it is not a third assumption to pass.
Time-average convergence explains when one long simulated path estimates the model's stationary probabilities.
We also distinguish that convergence from convergence of the state probabilities at a particular time.

> 요약: 맑음이면 1, 나머지는 0으로 기록하면 평균이 곧 맑은 날의 비율입니다. 방문 비율을 세는 이번 실습이 지난주의 시간평균입니다. 정상성·ACF·에르고딕성을 모두 통과해야만 시작하는 실습은 아닙니다.

| Step | What you do |
|---|---|
| 1 | Write down a transition matrix and step a distribution forward with `mu @ P` |
| 2 | Simulate the chain by hand (Monte Carlo) and count visits |
| 3 | Solve `πP = π` as a linear system |
| 4 | Compare the linear solve and `P^n`; allow sampling error in Monte Carlo |
| 5 | Watch `P^n` converge, and measure how fast |
| 6 | Break aperiodicity, then break irreducibility, and see which method still works |
| 7 | PageRank on a four-page web, with and without teleportation |

### How to use it

Two cells are marked **`TODO`**. Write those yourself first; they are the part worth doing by hand.
직접 채워 보는 것이 이 노트북의 핵심입니다.

> Each `TODO` is one line that reads `… = None`. Replace the `None` with your code; there is nothing to delete.
> If you run the cell before that, it stops with a `NotImplementedError` whose message says exactly this.
> **That is expected, not a broken notebook.** Each `TODO` is followed by a collapsed **Solution (정답)** cell:
> run it and the notebook continues from there. Open it only after you have tried.
> 각 `TODO` 는 `… = None` 한 줄입니다. `None` 자리에 코드를 넣으면 되고, 지울 줄은 없습니다.

> ### Before you touch anything: **File > Save a copy in Drive**
> The link I posted opens **read-only**. Colab will say *"changes will not be saved"*.
> Save your own copy first, or everything you type here disappears when you close the tab.
>
> **Nothing to submit this week.** Assignment 1 (12%) goes out after Week 4 and will use exactly this machinery,
> so the two TODO cells are the ones to get right now.

---
## 0. Setup

Nothing to install this week. Everything below ships with Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

SEED = 42                       # fixed seed = reproducible; every graded submission needs one
rng  = np.random.default_rng(SEED)

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (11, 3.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("numpy:", np.__version__)

### Optional. Korean labels in plots / 그림에 한글 쓰기

Every figure here is labelled in English, so you can skip this. But Colab ships **no Korean font**, so the moment you
write a Korean title yourself the characters come out as boxes. Run the cell below and it is fixed for the rest of
the session, **no runtime restart needed**.

이 노트북의 그림은 전부 영문이라 건너뛰어도 됩니다. 다만 Colab에는 한글 폰트가 없어서, 여러분이 한글 제목을 쓰는
순간 네모로 깨집니다. 아래 셀을 실행하면 **런타임 재시작 없이** 바로 해결됩니다.

In [ ]:
#@title ▶ 한글 폰트 설치 / Install Korean font (optional) { display-mode: "form" }
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1
import matplotlib.font_manager as fm

_p = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
try:
    fm.fontManager.addfont(_p)                  # 캐시 재생성 없이 즉시 등록 -> 재시작 불필요
    plt.rc("font", family=fm.FontProperties(fname=_p).get_name())
    plt.rc("axes", unicode_minus=False)         # 음수 눈금 깨짐 방지
    print("Korean font ready:", plt.rcParams["font.family"][0])
except Exception as _e:
    # apt 가 막혀도 노트북 전체가 멈추지 않도록 한다. 그림 라벨만 영문/네모로 나온다.
    plt.rc("axes", unicode_minus=False)
    print("한글 폰트를 건너뜁니다 (그림 라벨이 깨질 수 있습니다):", _e)

# 함정 1. 라벨에 유니코드 마이너스(U+2212)나 공집합(U+2205)을 직접 타이핑하면
#         이 폰트에 글리프가 없어 네모로 뜹니다. ASCII 하이픈(-)을 쓰세요.
# 함정 2. 로그 축 눈금(10^-3 등)은 mathtext 로 그려지므로 unicode_minus=False 로도 막히지 않습니다.
#         로그 축을 쓸 때는 FuncFormatter 로 눈금 문자열을 직접 만드세요:
#           from matplotlib.ticker import FuncFormatter
#           ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"1e{int(round(np.log10(y)))}"))


---
## 1. A chain, and one tick of the clock

Three weather states, **Sunny, Cloudy, Rainy**. Row *i* of `P` is the full conditional distribution of tomorrow
given that today is state *i*, so **every row must sum to 1** (deck: “The Transition Matrix P”).

The only dynamics you need all week: if `mu` is today's distribution as a row vector, tomorrow's is `mu @ P`.

### Same idea, a different weather model
The lecture's two-state example uses Sunny and Rainy and has sunny stationary probability two-thirds.
This lab uses **three** states and a different transition matrix. Its stationary probabilities are **21/46, 13/46, 12/46**.
Do not copy the lecture's two-thirds into this calculation.

We keep P fixed over time (time homogeneity). Starting with `mu = [1, 0, 0]` does not make the process stationary:
the first day's sunny probability is 1 and the next day's is 0.7. Starting instead from the stationary distribution would preserve the distribution at every step.

> 요약: 덱은 두 상태, 실습은 세 상태이므로 숫자는 다릅니다. 전이표가 일정한 시간동질성과 분포가 유지되는 정상성도 구분하세요. 덱 「One Weather Record, Two Weeks of Questions」·「The Weather Example Answers Last Week」.

In [ ]:
STATES = ["Sunny", "Cloudy", "Rainy"]

P = np.array([[0.70, 0.20, 0.10],      # from Sunny
              [0.30, 0.40, 0.30],      # from Cloudy
              [0.20, 0.30, 0.50]])     # from Rainy

print("row sums:", P.sum(axis=1))      # must be 1, 1, 1
assert np.allclose(P.sum(axis=1), 1), "rows must sum to 1; that is what 'stochastic matrix' means"

mu = np.array([1.0, 0.0, 0.0])         # certain it is Sunny today
print("\nday  " + "  ".join(f"{s:>7}" for s in STATES))
for day in range(6):
    print(f"{day:3d}  " + "  ".join(f"{p:7.4f}" for p in mu))
    mu = mu @ P

The distribution is approaching a fixed row. For this finite, irreducible, aperiodic chain, every initial distribution approaches the same stationary distribution.

That is one question. Another is whether visit frequencies along one long path approach that distribution.
The alternator later in the lab separates these two questions: its time averages converge even though its state probabilities can keep oscillating.

> 요약: 시각별 상태확률의 수렴과 한 경로의 방문 비율 수렴은 서로 다른 주장입니다. 뒤의 교대 연쇄에서 차이를 봅니다.

---
## 2. Route ①, simulate the chain

The most literal reading of the model: stand in a state, roll a die weighted by that row, move, repeat.
`rng.choice(3, p=P[s])` draws the next state from row `s`.

### The time average hidden in the counts
For each visited state, imagine writing 1 if it is Sunny and 0 otherwise.
The sum of these indicators is `counts[0]`; their mean is `counts[0] / n_steps`.
Repeat for Cloudy and Rainy to obtain all three visit frequencies.

For a finite irreducible chain, these frequencies approach the unique stationary distribution, even from a fixed starting state.
Aperiodicity is not required for this time-average conclusion.

> 요약: `counts[0] / n_steps`는 맑음 지시변수의 표본평균입니다. 유한 기약 연쇄의 장기 방문 비율 정리가 이 평균을 정당화합니다. 덱 「Three Ways to Compute the Stationary Distribution」.

In [ ]:
# ---------------------------------------------------------------- TODO ----
def simulate_chain(P, n_steps, rng, start=0):
    """Run the chain for n_steps and return the visit counts as an array of length len(P).

    counts[j] = how many of the n_steps visits landed in state j.
    """
    n = len(P)
    counts = np.zeros(n)
    s = start
    for _ in range(n_steps):
        s_next = None      # TODO: draw tomorrow's state from row s of P.   Hint: rng.choice(n, p=P[s])
        if s_next is None:
            raise NotImplementedError(
                "\n\n"
                "  정상입니다. 고장이 아니라 일부러 비워 둔 칸입니다.\n"
                "     위의 None 자리에 코드를 넣고 이 셀을 다시 실행하세요. 지울 줄은 없습니다.\n"
                "     막힐 때 바로 아래 'Solution / 정답' 셀을 실행하면 이어서 진행됩니다.\n\n"
                "  This is expected, not a broken notebook.\n"
                "     Replace None above with your code and re-run. Nothing needs deleting.\n"
                "     If stuck, run the Solution cell just below and continue.\n"
            )
        s = s_next
        counts[s] += 1     # given: count the visit
    return counts
# --------------------------------------------------------------------------


counts = simulate_chain(P, 200_000, np.random.default_rng(SEED), start=0)
pi_mc = counts / counts.sum()
print("Monte Carlo frequencies:", pi_mc)


In [ ]:
#@title ▶ Solution / 정답. Run only if you are stuck (this overwrites your version) { display-mode: "form" }
def simulate_chain(P, n_steps, rng, start=0):
    n = len(P)
    counts = np.zeros(n)
    s = start
    for _ in range(n_steps):
        s = rng.choice(n, p=P[s])
        counts[s] += 1
    return counts

counts = simulate_chain(P, 200_000, np.random.default_rng(SEED), start=0)
pi_mc = counts / counts.sum()
print("Monte Carlo frequencies:", pi_mc)

---
## 3. Route ②, solve πP = π

The deck's “The Stationary Distribution” defines π as the row vector with `πP = π` and `Σπ = 1`. Transposing turns the first part into the familiar
`Aπᵀ = 0` shape:

$$\pi P = \pi \iff (P^{\mathsf T} - I)\,\pi^{\mathsf T} = 0$$

Three equations, but only two of them are independent (each row of `P − I` sums to 0, so the rows of `Pᵀ − I` add up to the zero row), so the normalization
`Σπ = 1` is not an extra decoration; it is the equation that pins the answer down. Stack it on and solve the
4×3 least-squares system.

In [ ]:
# ---------------------------------------------------------------- TODO ----
def stationary_by_solve(P):
    """Return the stationary distribution as a 1-D array, by solving the linear system."""
    n = len(P)
    A = np.vstack([P.T - np.eye(n), np.ones(n)])   # n equations + the normalization row
    b = np.concatenate([np.zeros(n), [1.0]])
    pi = None          # TODO: solve A @ pi = b (least squares).   Hint: np.linalg.lstsq(A, b, rcond=None)[0]
    if pi is None:
        raise NotImplementedError(
            "\n\n"
            "  정상입니다. 고장이 아니라 일부러 비워 둔 칸입니다.\n"
            "     위의 None 자리에 코드를 넣고 이 셀을 다시 실행하세요. 지울 줄은 없습니다.\n"
            "     막힐 때 바로 아래 'Solution / 정답' 셀을 실행하면 이어서 진행됩니다.\n\n"
            "  This is expected, not a broken notebook.\n"
            "     Replace None above with your code and re-run. Nothing needs deleting.\n"
            "     If stuck, run the Solution cell just below and continue.\n"
        )
    return pi
# --------------------------------------------------------------------------


pi_solve = stationary_by_solve(P)
print("pi (linear solve):", pi_solve)
print("check pi @ P == pi:", np.allclose(pi_solve @ P, pi_solve))


In [ ]:
#@title ▶ Solution / 정답. Run only if you are stuck { display-mode: "form" }
def stationary_by_solve(P):
    n = len(P)
    A = np.vstack([P.T - np.eye(n), np.ones(n)])
    b = np.concatenate([np.zeros(n), [1.0]])
    return np.linalg.lstsq(A, b, rcond=None)[0]

pi_solve = stationary_by_solve(P)
print("pi (linear solve):", pi_solve)
print("check pi @ P == pi:", np.allclose(pi_solve @ P, pi_solve))

---
## 4. Raise P to a power, then compare all three routes

The deck's “Three Ways to Compute the Stationary Distribution” compares a linear solve, matrix powers and simulation.
For this weather chain, the first two agree numerically once the power is sufficiently large. Monte Carlo approximates the same vector with sampling error.

> 요약: 충분히 큰 행렬 거듭제곱과 선형해는 수치적으로 일치하고, 몬테카를로에는 표본오차가 남습니다.

In [ ]:
P50 = np.linalg.matrix_power(P, 50)
pi_power = P50[0]                      # any row will do; check that below

print("P^50 =\n", P50)
print("\nrows of P^50 identical to 4 decimals:", np.allclose(P50, P50[0], atol=1e-4))

print(f"\n{'':>8}{'Sunny':>9}{'Cloudy':>9}{'Rainy':>9}")
for name, v in [("Monte Carlo", pi_mc), ("linear solve", pi_solve), ("P^50 row", pi_power)]:
    print(f"{name:>12}" + "".join(f"{x:9.4f}" for x in v))

print("\nexact answer for this chain: 21/46, 13/46, 12/46 =",
      np.array([21/46, 13/46, 12/46]))
print("gap between the two exact methods (P^50 row vs linear solve):",
      np.max(np.abs(pi_power - pi_solve)))
print("gap between Monte Carlo and the exact answer:",
      np.max(np.abs(pi_mc - pi_solve)).round(5),
      "  (sampling error; standard error is about 0.002 at 200,000 steps)")

Read the two gaps separately.

* The two **exact** methods (a row of `P^50` and the linear solve) agree to about 1e-10. If they disagree, something is wrong.
* **Monte Carlo** is a sample estimate. At 200,000 steps its standard error is about 0.002 per state (the steps are
  correlated, so it is larger than the 0.001 you would get from independent draws). With the fixed seed the gap is
  0.0013; with other seeds a gap of 0.003 or 0.004 is common, and none of that is a bug. Sampling error has no hard
  ceiling, so treat a gap above roughly 0.01 as a warning sign rather than as proof of a bug.

When one route really is wrong, the usual cause is a transposed matrix (“Common Pitfalls” in the deck: we use
row-stochastic `P` and multiply on the *left*, `mu @ P`).

Reading π: the chain is Sunny about 46% of the long run, Cloudy 28%, Rainy 26%, regardless of today's weather.

---
## 5. How fast is "eventually"?

`P^n → π` is a limit, so the practical question is how many steps buy how much accuracy. Measure the largest
error left in any entry of `P^n` and plot it on a log scale.

In [ ]:
ns = np.arange(1, 31)
err = [np.abs(np.linalg.matrix_power(P, n) - pi_solve).max() for n in ns]

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.semilogy(ns, err, "o-", ms=4, color="#3b6ea5")
# Log ticks are drawn as 10^-3 via mathtext, whose minus sign is missing from the Korean font.
# Formatting them as plain text keeps the axis readable whether or not you ran the font cell.
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"1e{int(round(np.log10(y)))}"))
ax.set_xlabel("n"); ax.set_ylabel("max |P^n - pi|")
ax.set_title("Convergence of P^n to the stationary distribution")
plt.tight_layout(); plt.show()

eig = np.sort(np.abs(np.linalg.eigvals(P)))[::-1]
print("eigenvalues of P (by size):", eig)
print("the second one, |lambda_2| =", round(eig[1], 4), "- the error shrinks like this factor per step")
for n in [1, 5, 10, 20]:
    print(f"  n = {n:2d}   max error = {err[n-1]:.2e}")

A straight line on a log scale means geometric convergence, and the slope is set by the **second largest**
eigenvalue in absolute value (here about 0.47). Every step multiplies the remaining error by roughly that factor,
so ten steps already leave less than 0.001. Eigenvalue 1 belongs to π itself; everything else decays.

### Where last week's ACF fits
Encode Sunny as 1 and the other states as 0. In the stationary version of this finite irreducible aperiodic chain,
the ACF of that indicator fades with the lag. It describes the echo that affects the precision of its sample mean.
It is not another assumption that must be checked before running a Markov chain.

The lecture's **two-state** weather indicator has ACF `0.4**h` in stationarity. That formula does not automatically apply to this **three-state** matrix.
The matrix-convergence error plotted above is not itself an ACF plot.

> 요약: ACF는 이 실습에서도 맑음 여부의 시간 의존을 설명합니다. 위 그래프는 ACF가 아니라 행렬 수렴 오차입니다. 덱의 두 상태 ACF를 세 상태 모형에 그대로 쓰지 마세요.

---
## 6. Experiment A, break aperiodicity

The strict alternator from the deck's “Discussion. Why Aperiodicity?”: A → B → A → B, never staying put. Nothing here is random.

In [ ]:
P_alt = np.array([[0.0, 1.0],
                  [1.0, 0.0]])

print("P^n for the alternator:")
for n in [1, 2, 3, 10, 11]:
    print(f"  n = {n:2d}:", np.linalg.matrix_power(P_alt, n).ravel())

print("\npi from the linear solve:", stationary_by_solve(P_alt))
print("pi from Monte Carlo    :", (lambda c: c / c.sum())(
      simulate_chain(P_alt, 100_000, np.random.default_rng(1), start=0)))

Read the three answers carefully. They do **not** say the same thing.

- `P^n` flips between the identity and the swap forever. It never converges, exactly as the deck's “Discussion. Why Aperiodicity?” claims.
- The linear solve still returns (0.5, 0.5): a stationary distribution **exists**. Start from (0.5, 0.5) and you
  stay at (0.5, 0.5). The fixed point is real; it is just never reached from a corner.
- Monte Carlo also returns (0.5, 0.5), because the *time average* over a long path does converge even here.

So periodicity kills one of the three routes, not all three. "Existence" and "convergence of `P^n`" are different
claims about different objects (the same deck slide), and the time-average is a third object again.

### Why this does not contradict Week 2
If we choose the alternator's initial state with equal probabilities, it is stationary.
Its 0/1 indicator alternates, so its ACF does not fade, yet its average approaches one-half.
Last week's decaying-autocovariance condition was **sufficient**, not necessary, for mean-square convergence of the sample mean of a weakly stationary process.

> 요약: 메아리가 사라지면 평균 수렴을 보일 수 있지만, 평균 수렴에 반드시 메아리가 사라져야 하는 것은 아닙니다. 덱 「Discussion. Why Aperiodicity?」·「When Does P^n Actually Converge?」.

---
## 7. Experiment B, break irreducibility

Two closed groups that never talk to each other: states 0–1 form one world, states 2–3 another.

In [ ]:
P_split = np.array([[0.5, 0.5, 0.0, 0.0],
                    [0.5, 0.5, 0.0, 0.0],
                    [0.0, 0.0, 0.2, 0.8],
                    [0.0, 0.0, 0.8, 0.2]])

print("stationary candidates, all valid:")
for w in [1.0, 0.5, 0.0]:
    pi_w = np.array([w/2, w/2, (1-w)/2, (1-w)/2])
    print(f"  w = {w:.1f} -> {pi_w}   pi @ P == pi ? {np.allclose(pi_w @ P_split, pi_w)}")

for start in [0, 2]:
    c = simulate_chain(P_split, 50_000, np.random.default_rng(2), start=start)
    print(f"\nMonte Carlo started in state {start}: {c / c.sum()}")

Every mixture of the two worlds satisfies `πP = π`, so there is a whole **family** of stationary distributions and
no single "long-run behaviour" to speak of. Monte Carlo does not average over the family; it reports whichever
world you started in, and the answer changes with the starting state. That is what non-uniqueness looks like in practice.

One caveat worth keeping straight: reducible does **not** automatically mean non-unique. What creates the family here
is that both classes are *closed*. A chain with one closed class plus some transient states still has exactly one π,
the transient states simply get probability 0. You will see precisely that in the next section.

---
## 8. PageRank, the theorem shipped as a product

Four pages. A links to B and C; B links to C; C links back to A; D links to C and **nobody links to D**.
A random surfer clicks a uniformly chosen outgoing link. That is a Markov chain on pages, and PageRank is its
stationary distribution (deck: “PageRank, the Web's Stationary Distribution”).

In [ ]:
PAGES = ["A", "B", "C", "D"]
LINKS = {"A": ["B", "C"], "B": ["C"], "C": ["A"], "D": ["C"]}

M = np.zeros((4, 4))
for src, outs in LINKS.items():
    for dst in outs:
        M[PAGES.index(src), PAGES.index(dst)] = 1 / len(outs)
print("click matrix M (rows sum to 1):", M.sum(axis=1))

# Without teleportation: D is unreachable, so it is transient.
print("\nM^60 (no teleport), first row:", np.linalg.matrix_power(M, 60)[0])

# With teleportation: with probability 1-d, jump to a uniformly random page.
d = 0.85
G = d * M + (1 - d) / 4 * np.ones((4, 4))
pr = stationary_by_solve(G)

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(PAGES, pr, color="#3b6ea5", width=0.55)
ax.set_ylabel("PageRank"); ax.set_title("PageRank with teleportation (d = 0.85)")
plt.tight_layout(); plt.show()

for p, v in sorted(zip(PAGES, pr), key=lambda t: -t[1]):
    print(f"  {p}: {v:.4f}")
print(f"\nD gets exactly (1 - d)/4 = {(1 - d) / 4:.4f} - teleport traffic and nothing else")

Two things to take away.

- **Without teleportation** the chain is reducible: nobody links to D, so D is transient and `M^60` gives it
  probability 0, yet π is still unique, because {A, B, C} is the only closed class. This is the caveat from §7,
  made concrete.
- **With teleportation** every page reaches every other page in one hop, so the chain is irreducible and aperiodic
  by construction and the limit theorem applies. D's score is exactly `(1−d)/4 = 0.0375`: the floor that teleporting
  alone buys a page with no incoming links.

C wins (0.394) narrowly over A (0.373): C collects a link from all three other pages, but A is the only page C
points to, so A recycles most of C's score straight back. Importance is a fixed point, not a link count.

---
## 9. Your turn

Small changes, real answers. Do at least two.

1. Change the weather chain so Rainy is stickier (`P[2,2] = 0.8`, spreading the rest), re-solve, and say in one
   sentence which way π moved and why.
2. Add a fourth state "Snow" that can only be entered from Rainy and always goes back to Rainy next day.
   Is the chain still irreducible? Aperiodic? Check your answer against `P^n`.
3. Make an absorbing chain (the deck's {Playing, Won, Lost} from “Absorbing Chains, a Worked Miniature”) and use Monte Carlo to estimate the expected number
   of steps before absorption. The slide claims 5; do you get it?
4. In the PageRank example, add a link D ← A and recompute. Does D overtake B? Explain the ranking in one sentence.

Write one or two sentences under each thing you tried. The sentences are the point, not the numbers.

### Connect back to last week
In one sentence each, explain:
* Why counting sunny visits is a time average.
* Why a fixed transition table does not make a fixed Sunny start stationary.
* Why the alternator has a stable visit fraction without converging state probabilities.

> 요약: 계산 결과를 정상성·시간평균·상태확률 수렴의 구분으로 설명해 보세요.

Reading: Ross 11e §4.4 (Theorem 4.1) and §4.4.1; Week 2 deck “Weak Stationarity”, “Ergodicity” and “Three Things to Remember”.

In [ ]:
# Scratch space for section 9.


---
## 10. Environment record

Nothing to submit this week, but from Assignment 1 onward every submission ends with this cell; it is how a marker
reproduces your numbers. Run it once now so it is not new later.

In [ ]:
import sys, platform, datetime
try:
    import torch; gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
except Exception:
    gpu = "torch not loaded"

print("run at      :", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("python      :", sys.version.split()[0], "on", platform.system())
print("runtime     :", gpu)
print("seed        :", SEED)
print("numpy       :", np.__version__)